<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_6_Reducing_Duplication_and_Improving_Readability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
# Replace with link to your file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

transactions_df = pd.read_csv(transactions)
print(transactions_df)

   user_id  amount transaction_date     status
0      1.0  250.50        1 03 2025  completed
1      2.0    0.00        2 03 2025    pending
2      1.0  175.75        1 03 2025  completed
3      3.0   89.99        3 03 2025  completed
4      2.0  -10.00        2 03 2025     failed
5      4.0  320.00        4 03 2025  completed
6      NaN  150.00        1 03 2025  completed
7      1.0  200.00        5 03 2025  completed


In [ ]:
from typing import Dict

# User lookup for enrichment (simulating an API response)
user_lookup: Dict[int, str] = {
    1: "Alice Johnson",
    2: "Bob Smith",
    3: "Carol Williams",
    4: "David Brown"
}

In [ ]:
def safe_divide(numerator: float,
                denominator: float,
                default: float = 0.0
                ) -> float:

    """safely divide and return a default value if denominator is zero."""
    if denominator == 0:
        return default
    return numerator / denominator

In [ ]:
def add_normalized_amount(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize amount using a safe divide helper."""
    df = df.copy()
    max_amount = df['amount'].max()

    df['normalized_amount'] = df['amount'].apply(
        lambda x: safe_divide(x, max_amount, 0.0)
    )
    return df

if __name__ == "__main__":
    results_df = add_normalized_amount(transactions_df)
    print(results_df.head())

   user_id  amount transaction_date     status  normalized_amount
0      1.0  250.50        1 03 2025  completed           0.782813
1      2.0    0.00        2 03 2025    pending           0.000000
2      1.0  175.75        1 03 2025  completed           0.549219
3      3.0   89.99        3 03 2025  completed           0.281219
4      2.0  -10.00        2 03 2025     failed          -0.031250


# **Problem**

In [ ]:
import pandas as pd
from typing import Dict

def run_enrichment_pipeline(df: pd.DataFrame,
                            user_lookup: Dict[int, str]) -> pd.DataFrame:
    """Map names to user"""
    # Problem 1: Hardcoded "Magic Number"
    threshold=100
    df = df[df['amount'] > threshold].copy()
    # Problem 2: Hardcoded default string
    df['user_name'] = df['user_id'].map(user_lookup).fillna('Unknown')
    return df

def save_results(df: pd.DataFrame, user_lookup: Dict[int, str]) -> None:
    # Problem 3: Hardcoded Infrastructure
    """ Saving DataFrame to the database"""
    conn = sqlite3.connect("transactions_prod.db")
    df = run_enrichment_pipeline(df, user_lookup)
    df.to_sql("daily_summary", conn, if_exists="replace")

if __name__ == "__main__":
    save_results(transactions_df, user_lookup)

# **Fix: Defining Configurations**

In [ ]:
from dataclasses import dataclass

@dataclass
class PipelineConfig:
    db_path: str
    target_table: str
    min_amount: float
    default_user: str

In [ ]:
def run_enrichment_pipeline(
    df: pd.DataFrame,
    user_lookup: Dict[int, str],
    config: PipelineConfig
) -> pd.DataFrame:
    """Map names to users"""

    df = df[df['amount'] > config.min_amount].copy()
    df['user_name'] = df['user_id'].map(user_lookup).fillna(config.default_user)
    return df

def save_results(
    df: pd.DataFrame,
    user_lookup: Dict[int, str],
    config: PipelineConfig
) -> None:
    """Saving DataFrame to the database"""
    conn = sqlite3.connect(config.db_path)
    df = run_enrichment_pipeline(df, user_lookup, config)
    df.to_sql(config.target_table, conn, if_exists="replace")

In [ ]:
config = PipelineConfig(
    db_path="transactions_prod.db",
    target_table="daily_summary",
    min_amount=100,
    default_user="Unknown"
)

if __name__ == "__main__":
    save_results(transactions_df, user_lookup, config)

In [ ]:
config = {
    "db_path": "transactions_prod.db",
    "target_table": "daily_summary",
    "min_amount": 100,
    "default_user": "Unknown"
}

# **Challenge**

In [ ]:
import pandas as pd

def add_normalized_amount(df: pd.DataFrame) -> pd.DataFrame:
    """Add normalized amount column (amount / max_amount)."""
    df_copy = df.copy()
    max_amount = df_copy['amount'].max()

    # Safe division - avoid division by zero
    if max_amount == 0:
        df_copy['normalized'] = 0.0
    else:
        df_copy['normalized'] = df_copy['amount'] / max_amount
    return df_copy

def calculate_profit_margin(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate profit margin (profit / revenue)."""
    df_copy = df.copy()
    revenue = df_copy['amount'].sum()

    # Safe division - avoid division by zero (duplicated logic)
    if revenue == 0:
        df_copy['margin'] = 0.0
    else:
        df_copy['margin'] = df_copy['profit'] / revenue

    return df_copy

# **Solution**

In [ ]:
def save_divide(numerator:float,
                     denominator:float,
                     default: float = 0.0) -> float:

    if denominator == 0:
        return default
    else:
        return numerator/denominator

In [ ]:
def safe_divide(numerator:float,
                     denominator:float,
                     default: float = 0.0) -> float:
    """ Safely divide two numbers"""
    if denominator == 0:
        return default
    else:
        return numerator/denominator

def add_normalized_amount(df: pd.DataFrame) -> pd.DataFrame:
    """Add normalized amount column (amount / max_amount)."""
    df_copy = df.copy()
    max_amount = df_copy['amount'].max()

    df_copy['normalized'] = df_copy['amount'].apply(
        lambda x: safe_divide(x, max_amount, 0.0)
    )
    return df_copy

def calculate_profit_margin(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate profit margin (profit / revenue)."""
    df_copy = df.copy()
    df_copy['profit']=df_copy['amount'] * 0.4
    revenue = df_copy['amount'].sum()

    df_copy['margin'] = df_copy['profit'].apply(
        lambda x: safe_divide(x, revenue)
    )
    return df_copy

add_normalized_amount(transactions_df)

,user_id,amount,transaction_date,status,normalized
0,1.0,250.50,1 03 2025,completed,0.782813
1,2.0,0.00,2 03 2025,pending,0.000000
2,1.0,175.75,1 03 2025,completed,0.549219
3,3.0,89.99,3 03 2025,completed,0.281219
4,2.0,-10.00,2 03 2025,failed,-0.031250
5,4.0,320.00,4 03 2025,completed,1.000000
6,NaN,150.00,1 03 2025,completed,0.468750
7,1.0,200.00,5 03 2025,completed,0.625000


In [ ]:
calculate_profit_margin(transactions_df)

,user_id,amount,transaction_date,status,profit,margin
0,1.0,250.50,1 03 2025,completed,100.200,0.085187
1,2.0,0.00,2 03 2025,pending,0.000,0.000000
2,1.0,175.75,1 03 2025,completed,70.300,0.059767
3,3.0,89.99,3 03 2025,completed,35.996,0.030603
4,2.0,-10.00,2 03 2025,failed,-4.000,-0.003401
5,4.0,320.00,4 03 2025,completed,128.000,0.108821
6,NaN,150.00,1 03 2025,completed,60.000,0.051010
7,1.0,200.00,5 03 2025,completed,80.000,0.068013
